In [ ]:
import pandas as pd

# raw = pd.read_excel('./raw/US_Tax_Engagement_List_Detailed_Report_Jan_Wk2_FY26.xlsx', skiprows=6)
# raw.to_feather('./processed/US_Tax_Engagement_List_Detailed_Report_Jan_Wk2_FY26.feather')
raw = pd.read_feather('./processed/US_Tax_Engagement_List_Detailed_Report_Jan_Wk2_FY26.feather')

In [2]:
# Create a reduced data set for analysis
# Create a new dataframe with only the columns needed for filtering and analysis
required_columns = [
    'Client ID',
    'Client',
    'Account Channel',
    'Engagement ID',
    'Engagement Status', 
    'Engagement Service Code',
    'Release Date',
    'Engagement',
    'ANSR / Tech Revenue ETD',
    'ANSR / Tech Revenue FYTD',
    'TER ETD',
    'TER FYTD'
]

# Check if all required columns exist
missing_columns = [col for col in required_columns if col not in raw.columns]
if missing_columns:
    print(f"Warning: The following columns are missing from the dataframe: {missing_columns}")
    print("Available columns that might match:")
    for missing_col in missing_columns:
        matches = [col for col in raw.columns if missing_col.lower() in col.lower()]
        if matches:
            print(f"  For '{missing_col}': {matches[:5]}")  # Show first 5 matches
else:
    print("All required columns found!")
    reduced_df = raw[required_columns].copy()

# Coerce data types as needed
reduced_df['Client ID'] = pd.to_numeric(reduced_df['Client ID'], errors='coerce')
reduced_df['Engagement Service Code'] = reduced_df['Engagement Service Code'].astype(str)
reduced_df['Engagement ID'] = reduced_df['Engagement ID'].astype(str)
reduced_df['Engagement Status'] = reduced_df['Engagement Status'].astype(str)
reduced_df['ANSR / Tech Revenue ETD'] = pd.to_numeric(reduced_df['ANSR / Tech Revenue ETD'], errors='coerce')
reduced_df['Release Date'] = pd.to_datetime(reduced_df['Release Date'], errors='coerce')
reduced_df['Engagement'] = reduced_df['Engagement'].astype(str)


# reduced_df.to_excel('./review/reduced_engagement_list.xlsx', index=False)


All required columns found!


In [3]:
reduced_df.to_feather('./processed/reduced_coerced_engagement_list.feather')

# Private Client Criteria
1. Filter the column `Engagement Services Code` to include only `11420`
2. Filter the column `Release Date` for dates greater than `01/01/2024`
3. Filter the column `Engagement` to remove records containing text `pof` desregard upper/lower case
4. Filter (remove) engagements `ANSR / Tech Revenue ETD` for engagements that equal `0`
5. Filter `Engagement Status` to only include `Closing`, `Completed`, `Pre-Closing`, and `Released`

In [16]:
# Apply all filtering criteria for Total Addressable Revenue
initial_count = len(reduced_df)
previous_count = initial_count
print(f"Initial subset dataframe shape: {reduced_df.shape} ({initial_count:,} rows)")

# 1. Filter Engagement Services Code to include only 11420
df_filtered = reduced_df[reduced_df['Engagement Service Code'] == '11420']
current_count = len(df_filtered)
reduction_from_prev = previous_count - current_count
pct_reduction_from_prev = (reduction_from_prev / previous_count * 100) if previous_count > 0 else 0
print(f"After filtering Engagement Service Code = 11420: {df_filtered.shape} ({current_count:,} rows, {reduction_from_prev:,} rows removed, {pct_reduction_from_prev:.1f}% reduction from previous step)")
previous_count = current_count

# 2. Filter Release Date for dates greater than 01/01/2024
df_filtered = df_filtered[df_filtered['Release Date'] > '2024-01-01']
current_count = len(df_filtered)
reduction_from_prev = previous_count - current_count
pct_reduction_from_prev = (reduction_from_prev / previous_count * 100) if previous_count > 0 else 0
print(f"After filtering Release Date > 01/01/2024: {df_filtered.shape} ({current_count:,} rows, {reduction_from_prev:,} rows removed, {pct_reduction_from_prev:.1f}% reduction from previous step)")
previous_count = current_count

# 3. Filter Engagement column to exclude text containing 'pof' (case insensitive)
df_filtered = df_filtered[~df_filtered['Engagement'].str.contains('pof', case=False, na=False)]
current_count = len(df_filtered)
reduction_from_prev = previous_count - current_count
pct_reduction_from_prev = (reduction_from_prev / previous_count * 100) if previous_count > 0 else 0
print(f"After filtering to exclude Engagement containing 'pof': {df_filtered.shape} ({current_count:,} rows, {reduction_from_prev:,} rows removed, {pct_reduction_from_prev:.1f}% reduction from previous step)")
previous_count = current_count

# 4. Filter ANSR / Tech Revenue ETD for values greater than 0
df_filtered = df_filtered[df_filtered['ANSR / Tech Revenue ETD'] != 0]
current_count = len(df_filtered)
reduction_from_prev = previous_count - current_count
pct_reduction_from_prev = (reduction_from_prev / previous_count * 100) if previous_count > 0 else 0
print(f"After filtering ANSR / Tech Revenue ETD > 0: {df_filtered.shape} ({current_count:,} rows, {reduction_from_prev:,} rows removed, {pct_reduction_from_prev:.1f}% reduction from previous step)")
previous_count = current_count

# 5. Filter Engagement Status to include only specific statuses
allowed_statuses = ['Closing', 'Completed', 'Pre-Closing', 'Released']
df_filtered = df_filtered[df_filtered['Engagement Status'].isin(allowed_statuses)]
current_count = len(df_filtered)
reduction_from_prev = previous_count - current_count
pct_reduction_from_prev = (reduction_from_prev / previous_count * 100) if previous_count > 0 else 0
print(f"After filtering Engagement Status to {allowed_statuses}: {df_filtered.shape} ({current_count:,} rows, {reduction_from_prev:,} rows removed, {pct_reduction_from_prev:.1f}% reduction from previous step)")

print(f"\nFinal filtered dataframe shape: {df_filtered.shape}")
print(f"Reduction: {reduced_df.shape[0] - df_filtered.shape[0]:,} rows removed ({((reduced_df.shape[0] - df_filtered.shape[0]) / reduced_df.shape[0] * 100):.1f}% reduction)")

Initial subset dataframe shape: (210873, 12) (210,873 rows)
After filtering Engagement Service Code = 11420: (38434, 12) (38,434 rows, 172,439 rows removed, 81.8% reduction from previous step)
After filtering Release Date > 01/01/2024: (15799, 12) (15,799 rows, 22,635 rows removed, 58.9% reduction from previous step)
After filtering to exclude Engagement containing 'pof': (7522, 12) (7,522 rows, 8,277 rows removed, 52.4% reduction from previous step)
After filtering ANSR / Tech Revenue ETD > 0: (7146, 12) (7,146 rows, 376 rows removed, 5.0% reduction from previous step)
After filtering Engagement Status to ['Closing', 'Completed', 'Pre-Closing', 'Released']: (7146, 12) (7,146 rows, 0 rows removed, 0.0% reduction from previous step)

Final filtered dataframe shape: (7146, 12)
Reduction: 203,727 rows removed (96.6% reduction)


# Maestro

In [17]:
# Read Maestro engagement IDs and mark presence using left-hand data only
maestro_df = pd.read_excel('./raw/Maestro Engagement Data_Sept_2025.xlsx')
# Keep only the key and deduplicate to avoid creating multiple matches per left row
maestro_df = maestro_df[['Engagement Id']].drop_duplicates()



# Avoid a full merge that can duplicate left rows when the right-side has multiple matches.
# Instead, copy the left dataframe and add a boolean flag indicating membership in Maestro.
pcs_merged = df_filtered.copy()
# Ensure types align (both strings) to avoid false negatives
pcs_merged['Engagement ID'] = pcs_merged['Engagement ID'].astype(str)
maestro_ids = maestro_df['Engagement Id'].astype(str)
pcs_merged['In Maestro'] = pcs_merged['Engagement ID'].isin(maestro_ids)

# Calculate ETD totals
pcs_total_etd = pcs_merged['ANSR / Tech Revenue ETD'].nunique()
pcs_total_count = pcs_merged['ANSR / Tech Revenue ETD'].count()
pcs_maestro_etd = pcs_merged.loc[pcs_merged['In Maestro'], 'ANSR / Tech Revenue ETD'].nunique()
pcs_maestro_count = pcs_merged.loc[pcs_merged['In Maestro'], 'ANSR / Tech Revenue ETD'].count()


print(f"\nTotal PCS ANSR / Tech Revenue ETD: ${pcs_total_etd:,.2f}")
print(f"Maestro PCS ANSR / Tech Revenue ETD: ${pcs_maestro_etd:,.2f}")
print(f"Total Private Engagements: {pcs_total_count}")
print(f"Maestro Engagements: {pcs_maestro_count}")

# Save the left-only merged result (keeps only rows from the left dataframe)
pcs_merged.to_excel('./review/pcs_merged_engagements.xlsx', index=False)



Total PCS ANSR / Tech Revenue ETD: $6,567.00
Maestro PCS ANSR / Tech Revenue ETD: $2,344.00
Total Private Engagements: 7146
Maestro Engagements: 2369


In [18]:
# Search for Private Tax Engagement Service Codes 10459 and 11420
codes = ['10459', '11420']
combined = reduced_df[reduced_df['Engagement Service Code'].isin(codes)].copy()

for code in codes:
    df_code = reduced_df[reduced_df['Engagement Service Code'] == code].copy()
    count = len(df_code)
    total_etd = df_code['ANSR / Tech Revenue ETD'].sum()
    total_fytd = df_code.get('ANSR / Tech Revenue FYTD', pd.Series(dtype='float')).sum()
    print(f"Code {code}: {count:,} rows | ETD total: ${total_etd:,.2f} | FYTD total: ${total_fytd:,.2f}")
    # Save per-code review file
    df_code.to_excel(f'./review/engagement_service_code_{code}.xlsx', index=False)

# Combined summary
combined_count = len(combined)
combined_etd = combined['ANSR / Tech Revenue ETD'].sum()
combined_fytd = combined.get('ANSR / Tech Revenue FYTD', pd.Series(dtype='float')).sum()
print(f"\nCombined codes {codes}: {combined_count:,} rows | ETD total: ${combined_etd:,.2f} | FYTD total: ${combined_fytd:,.2f}")
combined.to_excel('./review/ftts_10459_11420_combined.xlsx', index=False)


Code 10459: 1,736 rows | ETD total: $295,184,045.14 | FYTD total: $32,649,304.69
Code 11420: 38,434 rows | ETD total: $780,749,970.76 | FYTD total: $129,092,955.19

Combined codes ['10459', '11420']: 40,170 rows | ETD total: $1,075,934,015.90 | FYTD total: $161,742,259.88


# Prodigy

In [19]:
research_and_development = reduced_df[reduced_df['Engagement Service Code'] == '10676']

# Apply all filtering criteria for Total Addressable Revenue
rd_initial_count = len(research_and_development)
rd_previous_count = rd_initial_count
print(f"Initial subset dataframe shape: {reduced_df.shape} ({initial_count:,} rows)")

# Filter Release Date for dates greater than 01/01/2024
# research_and_development = research_and_development[research_and_development['Release Date'] > '2024-01-01']
# rd_current_count = len(research_and_development)
# rd_reduction_from_prev = rd_previous_count - rd_current_count
# rd_pct_reduction_from_prev = (rd_reduction_from_prev / rd_previous_count * 100) if rd_previous_count > 0 else 0
# print(f"After filtering Release Date > 01/01/2024: {research_and_development.shape} ({rd_current_count:,} rows, {rd_reduction_from_prev:,} rows removed, {rd_pct_reduction_from_prev:.1f}% reduction from previous step)")
# rd_previous_count = rd_current_count

# Filter ANSR / Tech Revenue ETD for values greater than 0
# research_and_development = research_and_development[research_and_development['ANSR / Tech Revenue ETD'] != 0]
# rd_current_count = len(research_and_development)
# rd_reduction_from_prev = rd_previous_count - rd_current_count
# rd_pct_reduction_from_prev = (rd_reduction_from_prev / rd_previous_count * 100) if rd_previous_count > 0 else 0
# print(f"After filtering ANSR / Tech Revenue ETD > 0: {research_and_development.shape} ({rd_current_count:,} rows, {rd_reduction_from_prev:,} rows removed, {rd_pct_reduction_from_prev:.1f}% reduction from previous step)")
# rd_previous_count = rd_current_count

research_and_development.to_excel('./review/research_and_development_engagements_10676.xlsx', index=False)

Initial subset dataframe shape: (210873, 12) (210,873 rows)


# Vector 

In [20]:
# Codes for Vector 10675 and 10677
import numpy as np

vector_codes = ['10675', '10677']
vector_opp = reduced_df[reduced_df['Engagement Service Code'].isin(vector_codes)].copy()
vector_app = pd.read_excel('./raw/vector_clients_engagements.xlsx')

# vector_app['ClientCode'] = vector_app['ClientCode'].astype(int, errors='coerce').notna()
vector_app['ClientCode'] = vector_app['ClientCode'].apply(lambda x: int(x) if str(x).isdigit() else 0)
vector_app['Engagementcode'] = vector_app['Engagementcode'].astype(str)

vector_merged = vector_opp.copy()

# Additional Filter rules
# Channel 2 clients only
vector_merged = vector_merged[vector_merged['Account Channel'] == '2']
# Only engagements greater than 10000
vector_merged = vector_merged[vector_merged['ANSR / Tech Revenue ETD'] > 10000]
# Where release date is > 1/1/2024
vector_merged = vector_merged[vector_merged['Release Date'] > '2024-01-01']
# Engagement Status in Closing, Completed, Pre-Closing, Released
allowed_statuses = ['Closing', 'Completed', 'Pre-Closing', 'Released']
vector_merged = vector_merged[vector_merged['Engagement Status'].isin(allowed_statuses)]

# Indicate whether clients and engagements in vector_app exist in vector_opp
vector_merged['vector_engagement'] = np.where(vector_merged['Engagement ID'].isin(vector_app['Engagementcode'].astype(str)), "Yes", "No")
vector_merged['vector_client'] = np.where(vector_merged['Client ID'].isin(vector_app['ClientCode'].astype(int)), "Yes", "No")



vector_merged.to_excel('./review/vector_engagements_10675_10677_v2.xlsx', index=False)

In [21]:
travis_file = pd.read_excel('./raw/vector_from_travis_10_27_25.xlsx', sheet_name='engagements')
#  add ansr / Tech Revenue FYTD from vector_merged to travis_file based on Engagement ID
travis_file = travis_file.merge(reduced_df[['Engagement ID', 'ANSR / Tech Revenue FYTD']], left_on='Eng ID', right_on='Engagement ID', how='left')

travis_file.to_excel('./review/vector_from_travis_with_fytd_10_27_25.xlsx', index=False)

# ShareTrust